# 03 — Stability Metrics Explorer

Explore stability metrics on synthetic data to build intuition for the
analysis pipeline before applying it to real recordings.  
Covers: per-window metrics, unstable region detection, and register analysis.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from audio.pitch_detector import hz_to_cents, get_note_name
from analysis.stability import (
    compute_stability_metrics,
    identify_unstable_regions,
    register_analysis,
)
from visualization.plotter import (
    plot_pitch_contour,
    plot_stability_heatmap,
    plot_register_comparison,
)
from utils.config import SAMPLE_RATE, HOP_LENGTH, STABILITY_THRESHOLD

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

RNG = np.random.default_rng(42)

## 1. Build synthetic data

We construct a 12-second sequence with three distinct sections:

| Seconds | Pitch | Behaviour |
|---------|-------|-----------|
| 0–4     | B♭3 (233 Hz) | Stable long tone |
| 4–8     | A4 (440 Hz)  | Unstable — large random wobble |
| 8–12    | E♭5 (622 Hz) | Stable with gentle vibrato |

In [ ]:
def make_frames(freq_hz, duration, noise_cents=0.0, vibrato_cents=0.0, vibrato_rate=5.5):
    """Return (times, frequencies) for a synthetic tone segment."""
    n = int(duration * SAMPLE_RATE / HOP_LENGTH)
    t = np.arange(n) * HOP_LENGTH / SAMPLE_RATE
    cents = (
        RNG.normal(0, noise_cents, n)
        + vibrato_cents * np.sin(2 * np.pi * vibrato_rate * t)
    )
    freqs = freq_hz * 2 ** (cents / 1200)
    return t, freqs

BB3, A4, EB5 = 233.08, 440.0, 622.25

t1, f1 = make_frames(BB3, 4.0, noise_cents=3.0)
t2, f2 = make_frames(A4,  4.0, noise_cents=60.0)
t3, f3 = make_frames(EB5, 4.0, noise_cents=2.0, vibrato_cents=15.0)

# Concatenate with correct time offsets
times       = np.concatenate([t1, t1[-1] + HOP_LENGTH/SAMPLE_RATE + t2,
                               t1[-1] + t2[-1] + 2*HOP_LENGTH/SAMPLE_RATE + t3])
frequencies = np.concatenate([f1, f2, f3])

print(f'Total frames : {len(times)}')
print(f'Duration     : {times[-1]:.2f}s')

## 2. Pitch contour

In [ ]:
fig = plot_pitch_contour(times, frequencies, title='Synthetic Pitch Contour', save=False)
plt.show()

## 3. Per-window stability metrics

In [ ]:
metrics = compute_stability_metrics(times, frequencies, window_size=1.0)

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
w_centres = metrics['window_starts'] + 0.5

axes[0].bar(w_centres, metrics['variance'], width=0.85, color='steelblue')
axes[0].axhline(STABILITY_THRESHOLD**2, color='red', linewidth=0.8,
                linestyle='--', label=f'Threshold ({STABILITY_THRESHOLD}²={STABILITY_THRESHOLD**2} cents²)')
axes[0].set_ylabel('Variance (cents²)')
axes[0].legend(fontsize=8)

axes[1].bar(w_centres, metrics['drift_rate'], width=0.85, color='darkorange')
axes[1].axhline(0, color='gray', linewidth=0.6)
axes[1].set_ylabel('Drift rate (cents/s)')

axes[2].bar(w_centres, metrics['stability_score'], width=0.85,
            color=[plt.cm.RdYlGn(s) for s in metrics['stability_score']])
axes[2].set_ylim(0, 1.05)
axes[2].set_ylabel('Stability score')
axes[2].set_xlabel('Time (s)')

fig.suptitle('Per-Window Stability Metrics', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Stability heatmap

In [ ]:
fig = plot_stability_heatmap(times, frequencies, window_size=1.0, save=False)
plt.show()

## 5. Unstable region detection

In [ ]:
regions = identify_unstable_regions(times, frequencies, threshold=STABILITY_THRESHOLD)

print(f'Unstable regions found: {len(regions)}\n')
print(f'  {"Start":>7}  {"End":>7}  {"Note":>6}  {"Mean variance":>14}')
print('  ' + '-' * 40)
for start, end, note, var in regions:
    print(f'  {start:7.2f}s  {end:7.2f}s  {note:>6}  {var:14.1f} cents²')

# Overlay regions on the pitch contour
cents = np.array([hz_to_cents(f) for f in frequencies])
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(times, cents, color='steelblue', linewidth=0.8, label='Pitch (cents)')
for start, end, note, _ in regions:
    ax.axvspan(start, end, alpha=0.25, color='red', label='Unstable')
    ax.text((start + end) / 2, ax.get_ylim()[1] * 0.9, note,
            ha='center', fontsize=8, color='darkred')
ax.axhline(0, color='gray', linewidth=0.6, linestyle='--')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Cents (relative to A4)')
ax.set_title('Pitch Contour with Unstable Regions Highlighted')
handles = [mpatches.Patch(color='steelblue', label='Pitch'),
           mpatches.Patch(color='red', alpha=0.4, label='Unstable region')]
ax.legend(handles=handles, fontsize=8)
plt.tight_layout()
plt.show()

## 6. Register analysis

In [ ]:
reg = register_analysis(times, frequencies)

for name, data in reg.items():
    print(f'\n── {name.upper()} ──')
    print(f'  Frames        : {data["frame_count"]}')
    print(f'  Time fraction : {data["time_fraction"]*100:.1f}%')
    m = data['metrics']
    if m is not None:
        valid = m['stability_score'][~np.isnan(m['stability_score'])]
        if valid.size:
            print(f'  Mean stability: {valid.mean():.3f}')
    print(f'  Unstable regions: {len(data["unstable_regions"])}')

In [ ]:
fig = plot_register_comparison(reg, save=False)
plt.show()

## 7. Effect of threshold on detected regions

Sweep the variance threshold to see how many regions are flagged at each level.

In [ ]:
thresholds = np.arange(5, 105, 5)
region_counts = [
    len(identify_unstable_regions(times, frequencies, threshold=th))
    for th in thresholds
]

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(thresholds, region_counts, marker='o', color='steelblue')
ax.axvline(STABILITY_THRESHOLD, color='red', linewidth=0.8, linestyle='--',
           label=f'Default threshold ({STABILITY_THRESHOLD} cents)')
ax.set_xlabel('Variance threshold (cents)')
ax.set_ylabel('Unstable regions detected')
ax.set_title('Sensitivity to Threshold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()